In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

from src.dynamicProgramming.state import SystemState
from src.dynamicProgramming.courier import Courier
from src.dynamicProgramming.feature_extractor import FeatureExtractor
from src.dynamicProgramming.value_function import ValueFunction
from src.dynamicProgramming.dp_trainer import DPTrainer

data_path = Path("../data/features/olist/delivery_jobs_dataset.csv")

In [ ]:
# loading a sample dataset
jobs = pd.read_csv(
    data_path,
    parse_dates = [
        'ready_time',
        'due_date'
    ]
)

jobs_sample = jobs.sample(20, random_state = 7)
display(jobs_sample.head())

# initializing system state
start_time = datetime.now()
state = SystemState(current_time = start_time)

# initialize couriers
state.couriers = []
nb_couriers = 5
for i in range(nb_couriers):
    courier = Courier(
        courier_id = i,
        start_time = start_time
    )

    state.couriers.append(courier)

# initializing active jobs
state.active_jobs = []

for _, row in jobs_sample.iterrows():
    state.active_jobs.append(row.to_dict())

print(f"Number of active jobs: {len(state.active_jobs)}")

# initializing feature extractor
feature_extractor = FeatureExtractor()
features = feature_extractor.extract(state)
print(f"Extracted features [num_active_jobs, avg_route_length, max_route_length, avg_completed_jobs]: {features}")
num_features = len(features)

# initialize value function
value_function = ValueFunction(num_features)
value = value_function.predict(features)
print(f"Predicted state value: {value}")

# initializing DP trainer
trainer = DPTrainer(value_function)

# propagating next state
next_state = SystemState(current_time = start_time)
next_state.couriers = state.couriers.copy()
next_state.active_jobs = state.active_jobs[:-1]
print(f"Number of active jobs in the next state: {len(next_state.active_jobs)}")

# reward
reward = -1

# running DP trainer
trainer.update(state, reward, next_state)
print(f"Updated weights: {value_function.weights}")

new_value = value_function.predict(features)
print(f"New predicted values: {new_value}")

,job_id,order_id,seller_id,customer_id,pickup_lat,pickup_lng,delivery_lat,delivery_lng,ready_time,due_date,service_time_min,demand
12130,491796bba785a397014b993ef8006380,491796bba785a397014b993ef8006380,53e4c6e0f4312d4d2107a8c9cddf45cd,7ec2fa9e2ebba90c99f413e7b8c5f3cd,-22.740325,-46.896754,-7.562445,-34.999375,2018-01-26 12:21:39,2018-03-12,30,1.0
37049,82121ebedf0f27bfb35630958489c7ea,82121ebedf0f27bfb35630958489c7ea,fa1c13f2614d7b5c4749cbc52fecda94,d0296a9161a79996162a6a4f0835ed2d,-22.822137,-47.270335,-23.561190,-46.474486,2018-06-25 17:19:17,2018-07-05,30,1.0
3450,3449aa90c200ac98335249ce9f0e3151,3449aa90c200ac98335249ce9f0e3151,5a05a16bb50629ee31afab8a6d4c2674,672fd46db2cd92beb42ce95dded37e49,-29.178216,-51.148684,-29.999230,-53.500549,2018-04-04 03:10:22,2018-04-19,30,1.0
22418,23289a87ad267a113c014020edf79ea2,23289a87ad267a113c014020edf79ea2,4c03b9dd4c11ee2cb35c96c49efc9420,d7f5c766748d1e31c16f498e25bfa6f3,-23.211746,-46.762875,-20.892832,-47.585040,2017-07-06 10:23:41,2017-07-26,30,1.0
64654,5b55ee8a50b508709962ccc9eba0de7f,5b55ee8a50b508709962ccc9eba0de7f,6560211a19b47992c3666cc44a7e94c0,eb18708944fc7cd5c0eb37c6e34944e7,-23.652366,-46.755753,-22.922381,-43.629511,2018-07-19 10:25:18,2018-08-16,30,1.0


Number of active jobs: 20
Extracted features [num_active_jobs, avg_route_length, max_route_length, avg_completed_jobs]: [20.  0.  0.  0.]
Predicted state value: 0.0
Number of active jobs in the next state: 19
Updated weights: [-0.2  0.   0.   0. ]
New predicted values: -4.0
